## 0. Parameters

In [1]:
from pathlib import Path

# ----------------------------- inputs ---------------------------------------
FOOTPRINTS_PATH = Path("../inference/post/germany/footprints.shp")       # patch polygons, must carry `pred_prob`
LOCATORS_PATH   = Path("../inference/post/germany/locators.shp")         # locator seed polygons (same grid as footprints)
PRIORS_PATH     = Path("../data/priors_germany.gpkg")  # priors layer, must carry `fid` (step 7)

# ----------------------------- outputs --------------------------------------
OUTPUT_PATH     = Path("../inference/post/germany/final.gpkg")
EXTRAS_PATH     = Path("../inference/post/germany/extras.gpkg")  # predictions filtered out in steps 8-9

# ----------------------------- pipeline parameters --------------------------
# Probability filter applied before step 1.
PRED_PROB_THRESHOLD = 0.66

# Layers the pred_prob filter is applied to (`pred_prob >= 0.66` on both layers,
# applied before the extraction in step 1).
FILTER_LAYERS = ("footprints", "locators")

# Step 7 — discard polygons that could not be joined to a prior. Non-matching
# polygons get a NULL fid and are dropped in step 8 anyway.
DISCARD_NONMATCHING = False

# Fallback CRS for shapefiles missing a .prj
ASSUME_CRS = "EPSG:4326"


## Imports and data loading

The `.shp` files may come without their `.shx` sidecar — `SHAPE_RESTORE_SHX` lets GDAL
rebuild it on the fly. **Note:** the `.dbf` sidecar (attribute table) cannot be rebuilt;
`footprints` needs its `pred_prob` column for the filter and for step 6.

In [2]:
import os
os.environ["SHAPE_RESTORE_SHX"] = "YES"  # rebuild .shx if missing

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely import STRtree
from shapely.geometry import Polygon, MultiPolygon

print("geopandas", gpd.__version__, "| shapely", shapely.__version__)

geopandas 1.1.2 | shapely 2.1.2


In [3]:
def load_layer(path, name):
    gdf = gpd.read_file(path)
    if gdf.crs is None:                       # .prj missing → assume project CRS
        gdf = gdf.set_crs(ASSUME_CRS)
        print(f"[{name}] no CRS on file — assuming {ASSUME_CRS}")
    print(f"[{name}] {len(gdf):,} features | columns: {list(gdf.columns)}")
    return gdf

footprints = load_layer(FOOTPRINTS_PATH, "footprints")
locators   = load_layer(LOCATORS_PATH,   "locators")

[footprints] 278,781 features | columns: ['pred', 'pred_prob', 'n_loc', 'geometry']
[locators] 77,624 features | columns: ['pred', 'pred_prob', 'geometry']


### Apply the `pred_prob` filter

The filter (`pred_prob >= 0.66`) is applied to **both** the footprints and locators layers
before the extraction in step 1. Note this also means step 6's `pred_prob_max` summary only
sees footprints above the threshold.

In [4]:
def prob_filter(gdf, name):
    if "pred_prob" not in gdf.columns:
        raise ValueError(
            f"'{name}' has no `pred_prob` column — the .dbf sidecar of the shapefile "
            "is required to apply the probability filter."
        )
    out = gdf[gdf["pred_prob"] >= PRED_PROB_THRESHOLD].reset_index(drop=True)
    print(f"[{name}] pred_prob >= {PRED_PROB_THRESHOLD}: {len(gdf):,} → {len(out):,}")
    return out

if "footprints" in FILTER_LAYERS:
    footprints = prob_filter(footprints, "footprints")
if "locators" in FILTER_LAYERS:
    locators = prob_filter(locators, "locators")

[footprints] pred_prob >= 0.66: 278,781 → 203,114
[locators] pred_prob >= 0.66: 77,624 → 54,967


## Step 1 — Extract by location (*overlap* OR *equal*)

Extract features from **footprints** where they **overlap** or are **equal** to features
from **locators**. The two predicates are OR-ed.

- *overlap*: interiors intersect, but neither geometry contains the other (GEOS `overlaps`)
- *equal*: geometrically identical (GEOS `equals`)

In [5]:
loc_geoms = locators.geometry.values
fp_geoms  = footprints.geometry.values
tree      = STRtree(loc_geoms)

# footprints that OVERLAP any locator
i_over, _ = tree.query(fp_geoms, predicate="overlaps")
selected  = set(i_over.tolist())

# footprints EQUAL to any locator (candidate pairs via intersects, then exact test —
# STRtree has no native 'equals' predicate)
i_cand, j_cand = tree.query(fp_geoms, predicate="intersects")
eq_mask = shapely.equals(fp_geoms[i_cand], loc_geoms[j_cand])
selected |= set(i_cand[eq_mask].tolist())

extracted = footprints.iloc[sorted(selected)].reset_index(drop=True)
print(f"extracted: {len(extracted):,}")

extracted: 193,083


## Step 2 — Dissolve

Dissolve with no dissolve fields and disjoint features **not** kept separate → all
geometries are unioned into a single (multi)polygon feature.

In [6]:
dissolved = shapely.unary_union(extracted.geometry.values)
print("dissolved geometry type:", dissolved.geom_type)

dissolved geometry type: MultiPolygon


## Step 3 — Delete holes

Interior rings are removed with a minimum area of `0.0` → **all** holes go; each polygon
is rebuilt from its exterior ring only.

In [7]:
def delete_holes(geom):
    """Rebuild polygon(s) from exterior rings only (all holes removed, min area = 0)."""
    if geom.geom_type == "Polygon":
        return Polygon(geom.exterior)
    return MultiPolygon([Polygon(p.exterior) for p in geom.geoms])

cleaned = delete_holes(dissolved)
n_parts = len(cleaned.geoms) if cleaned.geom_type == "MultiPolygon" else 1
print(f"cleaned: {cleaned.geom_type} with {n_parts:,} parts")

cleaned: MultiPolygon with 20,550 parts


## Step 4 — Multipart to singleparts

Explode the multipolygon into one feature per part.

In [8]:
parts = list(cleaned.geoms) if cleaned.geom_type == "MultiPolygon" else [cleaned]
single_parts = gpd.GeoDataFrame(geometry=parts, crs=footprints.crs)
print(f"single parts: {len(single_parts):,}")

single parts: 20,550


## Step 5 — Dissolve intersecting groups

Polygons that intersect directly **or transitively** (A–B, B–C ⇒ A+B+C) are merged into
single features via a graph traversal over an intersection spatial index; spatially
separate clusters stay separate. Needed here because hole-filling (step 3) makes formerly
nested polygons overlap, and `unary_union` alone keeps point-touching parts as separate
parts.

Invalid geometries are repaired with `make_valid`, each group is merged with `unary_union`,
and the output carries `group_id` and `n_parts`.

In [9]:
def dissolve_intersecting_groups(gdf):
    """Merge polygons that intersect directly or transitively into single features."""
    geoms = gdf.geometry.values.copy()

    # --- fix validity up front (make_valid on invalid input) ---
    invalid = ~shapely.is_valid(geoms)
    if invalid.any():
        geoms[invalid] = shapely.make_valid(geoms[invalid])

    # --- union-find over intersecting polygons (transitive) ---
    tree    = STRtree(geoms)
    n       = len(geoms)
    visited = np.zeros(n, dtype=bool)
    groups  = []
    for start in range(n):
        if visited[start]:
            continue
        stack, group = [start], []
        while stack:
            i = stack.pop()
            if visited[i]:
                continue
            visited[i] = True
            group.append(i)
            # bbox candidates + exact intersects test (includes touching polygons)
            for j in tree.query(geoms[i], predicate="intersects"):
                if not visited[j]:
                    stack.append(int(j))
        groups.append(group)

    # --- dissolve each group via unary union ---
    records = []
    for gid, group in enumerate(groups, start=1):
        merged = shapely.unary_union(geoms[group])
        if merged is None or merged.is_empty:
            continue
        if not merged.is_valid:
            merged = shapely.make_valid(merged)
        records.append({"group_id": gid, "n_parts": len(group), "geometry": merged})

    return gpd.GeoDataFrame(records, crs=gdf.crs)

dissolved_groups = dissolve_intersecting_groups(single_parts)
print(f"dissolved groups: {len(dissolved_groups):,}")

dissolved groups: 20,252


## Step 6 — Join attributes by location (summary)

Join to **dissolved groups** where they **contain** features from **footprints**,
summarising field `pred_prob` with statistic `max` → new column **`pred_prob_max`**.
Groups containing no footprint get NULL.

In [10]:
fp_tree = STRtree(footprints.geometry.values)
gi, fj  = fp_tree.query(dissolved_groups.geometry.values, predicate="contains")

max_per_group = (
    pd.Series(footprints["pred_prob"].values[fj], index=gi)
      .groupby(level=0).max()
)
dissolved_groups["pred_prob_max"] = dissolved_groups.index.map(max_per_group)
print(f"groups with pred_prob_max: {dissolved_groups['pred_prob_max'].notna().sum():,} "
      f"/ {len(dissolved_groups):,}")

groups with pred_prob_max: 20,252 / 20,252


## Step 7 — Join attributes by location (one-to-many, priors)

Join to the previous layer where features **intersect** `priors_germany`, adding field
**`fid`**, with a separate feature created for each matching prior (one-to-many) — a
polygon that intersects several priors is duplicated once per prior.

In [11]:
priors = load_layer(PRIORS_PATH, "priors_germany")

# In a GeoPackage, `fid` is the feature ID rather than an attribute column — read it
# explicitly if it wasn't returned as a column.
if "fid" not in priors.columns:
    priors = gpd.read_file(PRIORS_PATH, fid_as_index=True).reset_index(names="fid")
    if priors.crs is None:
        priors = priors.set_crs(ASSUME_CRS)
    print(f"[priors_germany] using GeoPackage feature ids as `fid`")

p_tree = STRtree(priors.geometry.values)
gi, pj = p_tree.query(dissolved_groups.geometry.values, predicate="intersects")

joined = dissolved_groups.iloc[gi].copy()
joined["fid"] = priors["fid"].values[pj]

if not DISCARD_NONMATCHING:
    # non-matching features are kept once, with NULL in the joined field
    unmatched = dissolved_groups.loc[~dissolved_groups.index.isin(gi)].copy()
    unmatched["fid"] = pd.NA
    joined = pd.concat([joined, unmatched])

joined = joined.reset_index(drop=True)
print(f"joined: {len(joined):,}")

[priors_germany] 29,133 features | columns: ['confidence', 'geometry']
[priors_germany] using GeoPackage feature ids as `fid`
joined: 20,485


## Step 8 — Keep the best polygon per prior

Within each prior `fid`, keep only the row(s) whose `pred_prob_max` equals the maximum
`pred_prob_max` for that `fid`. Rows where either side of the comparison is NULL evaluate
to false and are dropped.

In [12]:
group_max = joined.groupby("fid", dropna=False)["pred_prob_max"].transform("max")
matching  = joined[joined["pred_prob_max"] == group_max].reset_index(drop=True)
print(f"matching: {len(matching):,}")

matching: 18,302


## Step 9 — Delete duplicate geometries

A polygon that is the maximum for several priors appears once per prior after step 8 —
keep only the first occurrence of each geometry.

In [13]:
wkb   = shapely.to_wkb(shapely.normalize(matching.geometry.values))
final = matching.loc[~pd.Series(wkb, index=matching.index).duplicated()].reset_index(drop=True)
print(f"final: {len(final):,} features")

final: 18,115 features


## Export

Only `pred_prob_max` is kept in the output (helper columns `group_id`, `n_parts`, `fid`
are dropped). Geometries are written as MultiPolygon, matching the output layer type.

In [14]:
final_out = final[["pred_prob_max", "geometry"]].copy()
# match the gpkg's MultiPolygon layer type
final_out["geometry"] = final_out.geometry.apply(
    lambda g: MultiPolygon([g]) if g.geom_type == "Polygon" else g
)
final_out.to_file(OUTPUT_PATH, layer="final", driver="GPKG")
print(f"written: {OUTPUT_PATH} ({len(final_out):,} features)")

written: ../inference/post/germany/final.gpkg (18,115 features)


## Extras — predictions filtered out of the main pipeline

Dissolved-group polygons (with their `pred_prob_max`) that did **not** survive steps 8–9:
either they intersect no prior (NULL `fid`), or another polygon on the same prior had a
higher `pred_prob_max`. Saved separately for review.

In [15]:
final_wkb  = set(shapely.to_wkb(shapely.normalize(final.geometry.values)))
groups_wkb = shapely.to_wkb(shapely.normalize(dissolved_groups.geometry.values))
is_extra   = np.array([w not in final_wkb for w in groups_wkb])

extras = dissolved_groups.loc[is_extra, ["pred_prob_max", "geometry"]].reset_index(drop=True)
extras["geometry"] = extras.geometry.apply(
    lambda g: MultiPolygon([g]) if g.geom_type == "Polygon" else g
)
extras.to_file(EXTRAS_PATH, layer="extras", driver="GPKG")
print(f"written: {EXTRAS_PATH} ({len(extras):,} features filtered out)")

written: ../inference/post/germany/extras.gpkg (2,137 features filtered out)
